4

In [65]:
def calculate_dense_flops_forward(tokens: int, params: int):
    return 2.0 * tokens *  params


def calculate_dense_flops_backward(tokens: int, params: int):
    return 4.0 * tokens *  params
    

def calculate_mha_tflops(
    batch_size: int,
    query_seq_len: int,
    kv_seq_len: int,
    num_heads: int, 
    head_dim: int,
    num_segments: int,
    generation: bool,
):
    causal: bool = False if generation else True
    
    if generation and num_segments > 1:
        raise ValueError()
    
    if num_segments > 1:
        query_seq_len //= num_segments
        kv_seq_len //= num_segments

    flops = 4.0 * batch_size * head_dim * query_seq_len * kv_seq_len * num_heads
    
    flops *= num_segments
    if causal:
        flops *= 0.5
    return flops


class Layer:
    def __init__(self, generation: bool):
        self.generation = generation
    
    def forward(self, batch_size: int, seq_len: int):
        raise NotImplementedError()
    
    def backward(self, batch_size: int, seq_len: int):
        raise NotImplementedError()
    
    def params(self):
        raise NotImplementedError()


class Attention(Layer):
    def __init__(
        self,
        hidden_size: int,
        num_heads: int,
        num_kv_heads: int,
        head_dim: int,
        generation: bool,
    ):
        super().__init__(generation)
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        
        self.query_params = self.hidden_size * self.num_heads * self.head_dim
        self.kv_params = self.hidden_size * self.num_kv_heads * self.head_dim
        self.output_params = self.hidden_size * self.num_heads * self.head_dim

    def forward(self, batch_size: int, seq_len: int):
        query_seq_len = 1 if self.generation else seq_len 
        query_flops = calculate_dense_flops_forward(batch_size * query_seq_len, self.query_params)
        key_flops = calculate_dense_flops_forward(batch_size * query_seq_len, self.kv_params)
        value_flops = calculate_dense_flops_forward(batch_size * query_seq_len, self.kv_params)
        output_flops = calculate_dense_flops_forward(batch_size * query_seq_len, self.query_params)
        
        attn_flops = calculate_mha_tflops(
            batch_size,
            query_seq_len,
            seq_len,
            self.num_heads,
            self.head_dim,
            generation=self.generation,
            num_segments=12,
        )

        return sum([query_flops, key_flops, value_flops, output_flops, attn_flops])
        
    def backward(self, batch_size: int, seq_len: int):
        if self.generation:
            raise ValueError()
        query_flops = calculate_dense_flops_backward(batch_size * seq_len, self.query_params)
        key_flops = calculate_dense_flops_backward(batch_size * seq_len, self.kv_params)
        value_flops = calculate_dense_flops_backward(batch_size * seq_len, self.kv_params)
        output_flops = calculate_dense_flops_backward(batch_size * seq_len, self.query_params)
        
        attn_flops = calculate_mha_tflops(
            batch_size,
            seq_len,
            seq_len,
            self.num_heads,
            self.head_dim,
            generation=False,
            num_segments=12,
        ) * 2.5
        return sum([query_flops, key_flops, value_flops, output_flops, attn_flops])
    
    def params(self):
        return self.query_params + 2 * self.kv_params + self.output_params


class MLP(Layer):
    def __init__(
        self,
        hidden_size: int,
        intermediate_size: int,
        generation: bool,
    ):
        super().__init__(generation)
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        
        self.up_proj_params = self.hidden_size * self.intermediate_size
        self.gate_params = self.hidden_size * self.intermediate_size
        self.down_proj_params = self.intermediate_size * self.hidden_size

    def forward(self, batch_size: int, seq_len: int):
        query_seq_len = 1 if self.generation else seq_len 
        up_proj_flops = calculate_dense_flops_forward(batch_size * query_seq_len, self.up_proj_params)
        gate_flops = calculate_dense_flops_forward(batch_size * query_seq_len, self.gate_params)
        down_proj_flops = calculate_dense_flops_forward(batch_size * query_seq_len, self.down_proj_params)

        return sum([up_proj_flops, gate_flops, down_proj_flops])
    
    def backward(self, batch_size: int, seq_len: int):
        if self.generation:
            raise ValueError()
        up_proj_flops = calculate_dense_flops_backward(batch_size * seq_len, self.up_proj_params)
        gate_flops = calculate_dense_flops_backward(batch_size * seq_len, self.gate_params)
        down_proj_flops = calculate_dense_flops_backward(batch_size * seq_len, self.down_proj_params)

        return sum([up_proj_flops, gate_flops, down_proj_flops])
    
    def params(self):
        return self.up_proj_params + self.gate_params + self.down_proj_params


class Unembedding(Layer):
    def __init__(
        self,
        hidden_size: int,
        vocab_size: int,
        generation: bool,
    ):
        super().__init__(generation)
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size
        
        self.unembedding_params = self.hidden_size * self.vocab_size

    def forward(self, batch_size: int, seq_len: int):
        query_seq_len = 1 if self.generation else seq_len 
        return calculate_dense_flops_forward(batch_size * query_seq_len, self.unembedding_params)
    
    def backward(self, batch_size: int, seq_len: int):
        if self.generation:
            raise ValueError()
        return calculate_dense_flops_backward(batch_size * seq_len, self.unembedding_params)
    
    def params(self):
        return self.unembedding_params


class Normilizer(Layer):
    def __init__(
        self,
        hidden_size: int,
        generation: bool,
    ):
        super().__init__(generation)
        self.hidden_size = hidden_size
        
        self.norm_params = self.hidden_size

    def forward(self, batch_size: int, seq_len: int):
        query_seq_len = 1 if self.generation else seq_len 
        return 2.0 * self.hidden_size * batch_size * seq_len
    
    def backward(self, batch_size: int, seq_len: int):
        if self.generation:
            raise ValueError()
        return self.forward(batch_size, seq_len)
    
    def params(self):
        return self.norm_params


def calc_forward_flops(
    batch_size: int,
    seq_len: int,
    hidden_size: int,
    num_heads: int,
    num_kv_heads: int,
    head_dim: int,
    intermediate_size: int,
    vocab_size: int,
    num_layers: int,
    generation: bool,
):
    attn = Attention(hidden_size=hidden_size, num_heads=num_heads, num_kv_heads=num_kv_heads, head_dim=head_dim, generation=generation)
    mlp = MLP(hidden_size=hidden_size, intermediate_size=intermediate_size, generation=generation)
    unembedding = Unembedding(hidden_size=hidden_size, vocab_size=vocab_size, generation=generation)
    
    print(attn.forward(batch_size, seq_len))
    print(mlp.forward(batch_size, seq_len))
    print(unembedding.forward(batch_size, seq_len))
    
    return (attn.forward(batch_size, seq_len) + mlp.forward(batch_size, seq_len)) * num_layers + unembedding.forward(batch_size, seq_len)


def calc_backward_flops(
    batch_size: int,
    seq_len: int,
    hidden_size: int,
    num_heads: int,
    num_kv_heads: int,
    head_dim: int,
    intermediate_size: int,
    vocab_size: int,
    num_layers: int,
):
    attn = Attention(hidden_size=hidden_size, num_heads=num_heads, num_kv_heads=num_kv_heads, head_dim=head_dim, generation=False)
    mlp = MLP(hidden_size=hidden_size, intermediate_size=intermediate_size, generation=False)
    unembedding = Unembedding(hidden_size=hidden_size, vocab_size=vocab_size, generation=False)
    
    print(attn.forward(batch_size, seq_len), attn.backward(batch_size, seq_len))
    print(mlp.forward(batch_size, seq_len), mlp.backward(batch_size, seq_len))
    print(unembedding.forward(batch_size, seq_len), unembedding.backward(batch_size, seq_len))
    
    return (attn.backward(batch_size, seq_len) + mlp.backward(batch_size, seq_len)) * num_layers + unembedding.backward(batch_size, seq_len)

def num_params(
    hidden_size: int,
    num_heads: int,
    num_kv_heads: int,
    head_dim: int,
    intermediate_size: int,
    vocab_size: int,
    num_layers: int,
):
    attn = Attention(hidden_size=hidden_size, num_heads=num_heads, num_kv_heads=num_kv_heads, head_dim=head_dim, generation=False)
    mlp = MLP(hidden_size=hidden_size, intermediate_size=intermediate_size, generation=False)
    unembedding = Unembedding(hidden_size=hidden_size, vocab_size=vocab_size, generation=False)
    norm = Normilizer(hidden_size=hidden_size, generation=False)
    
    return (attn.params() + mlp.params() + norm.params()) * num_layers + unembedding.params() * 2 + norm.params()



In [66]:
# LLama 3.1 8b
fwd_flops = calc_forward_flops(
    batch_size=1,
    seq_len=8192,
    hidden_size=4096,
    num_heads=32,
    num_kv_heads=8,
    head_dim=128,
    intermediate_size=14336,
    vocab_size=128256,
    num_layers=32,
    generation=False,
)
print(f"{fwd_flops / 1e12:.3} TFLOPs")

bwd_flops = calc_backward_flops(
    batch_size=1,
    seq_len=8192,
    hidden_size=4096,
    num_heads=32,
    num_kv_heads=8,
    head_dim=128,
    intermediate_size=14336,
    vocab_size=128256,
    num_layers=32,
)
print(f"{bwd_flops / 1e12:.3} TFLOPs")

params = num_params(
    hidden_size=4096,
    num_heads=32,
    num_kv_heads=8,
    head_dim=128,
    intermediate_size=14336,
    vocab_size=128256,
    num_layers=32,
)
print("Params:", params)

732918317056.0
2886218022912.0
8607114461184.0
1.24e+02 TFLOPs
732918317056.0 1488698408960.0
2886218022912.0 5772436045824.0
8607114461184.0 17214228922368.0
2.5e+02 TFLOPs
Params: 8030130176


In [61]:
# Qwen 72b
# https://huggingface.co/Qwen/Qwen2.5-72B-Instruct/blob/main/config.json

fwd_flops = calc_forward_flops(
    batch_size=1,
    seq_len=8192,
    hidden_size=8192,
    num_heads=64,
    num_kv_heads=8,
    head_dim=128,
    intermediate_size=29568,
    vocab_size=152064,
    num_layers=80,
    generation=False,
)
print(f"{fwd_flops / 1e12:.3} TFLOPs")

bwd_flops = calc_backward_flops(
    batch_size=1,
    seq_len=8192,
    hidden_size=8192,
    num_heads=64,
    num_kv_heads=8,
    head_dim=128,
    intermediate_size=29568,
    vocab_size=152064,
    num_layers=80,
)
print(f"{bwd_flops / 1e12:.3} TFLOPs")
print(f"{(fwd_flops + bwd_flops) / 1e15:.3} PFLOPs")

3573412790272.0
11905649344512.0
20409684590592.0
1.26e+03 TFLOPs
3573412790272.0 7696581394432.0
11905649344512.0 23811298689024.0
20409684590592.0 40819369181184.0
2.56e+03 TFLOPs
3.82 PFLOPs


In [62]:
# Qwen 72b
# https://huggingface.co/Qwen/Qwen2.5-72B-Instruct/blob/main/config.json

fwd_flops = calc_forward_flops(
    batch_size=1,
    seq_len=65536,
    hidden_size=8192,
    num_heads=64,
    num_kv_heads=8,
    head_dim=128,
    intermediate_size=29568,
    vocab_size=152064,
    num_layers=80,
    generation=False,
)
print(f"{fwd_flops / 1e12:.3} TFLOPs")

bwd_flops = calc_backward_flops(
    batch_size=1,
    seq_len=65536,
    hidden_size=8192,
    num_heads=64,
    num_kv_heads=8,
    head_dim=128,
    intermediate_size=29568,
    vocab_size=152064,
    num_layers=80,
)
print(f"{bwd_flops / 1e12:.3} TFLOPs")
print(f"{(fwd_flops + bwd_flops) / 1e15:.3} PFLOPs")

90159953477632.0
95245194756096.0
163277476724736.0
1.5e+04 TFLOPs
90159953477632.0 215504279044096.0
95245194756096.0 190490389512192.0
163277476724736.0 326554953449472.0
3.28e+04 TFLOPs
47.8 PFLOPs


In [67]:
# Qwen 72b 12 segms
# https://huggingface.co/Qwen/Qwen2.5-72B-Instruct/blob/main/config.json

fwd_flops = calc_forward_flops(
    batch_size=1,
    seq_len=65536,
    hidden_size=8192,
    num_heads=64,
    num_kv_heads=8,
    head_dim=128,
    intermediate_size=29568,
    vocab_size=152064,
    num_layers=80,
    generation=False,
)
print(f"{fwd_flops / 1e12:.3} TFLOPs")

bwd_flops = calc_backward_flops(
    batch_size=1,
    seq_len=65536,
    hidden_size=8192,
    num_heads=64,
    num_kv_heads=8,
    head_dim=128,
    intermediate_size=29568,
    vocab_size=152064,
    num_layers=80,
)
print(f"{bwd_flops / 1e12:.3} TFLOPs")
print(f"{(fwd_flops + bwd_flops) / 1e15:.3} PFLOPs")

25654555508736.0
95245194756096.0
163277476724736.0
9.84e+03 TFLOPs
25654555508736.0 54240784121856.0
95245194756096.0 190490389512192.0
163277476724736.0 326554953449472.0
1.99e+04 TFLOPs
29.7 PFLOPs
